In [1]:
import sys
from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation_functional as sim
import xobjects as xo

In [2]:
n_turns = 200

line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

# line = sim.add_dipole(line, ...)

gamx = 0.13112848113527537 # gamma at 0
Jx_max = gamx * 0.05**2
Jx = np.linspace(0, Jx_max, 500)
x = np.sqrt(Jx / gamx)
p = np.zeros_like(x)
y = np.full_like(a=x, fill_value=0.005)
particles = line.build_particles(
    x=x+line.twiss(method='4d', freeze_longitudinal=True).x[0],
    y=y+line.twiss(method='4d', freeze_longitudinal=True).y[0],
    px=p+line.twiss(method='4d', freeze_longitudinal=True).px[0],
    py=p+line.twiss(method='4d', freeze_longitudinal=True).py[0],
                         method='4d', freeze_longitudinal=True)

line = sim.add_monitors(line, particles, n_monitors=100, n_turns=n_turns)

twiss = line.twiss(method='4d', freeze_longitudinal=True)

line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=5e12,
                                             n_interactions=100, 
                                             gemitt=60e-6, indirect=False)

line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
line.track(particles, num_turns=n_turns, time=True)


Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/4570 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [3]:
sim.save(line, particles, twiss, 
         filename='data/functional_test', 
         aperture_df=aper_df, beam_df=beam_df)

File saved at data/functional_test
